# Two-round 1000G-referenced classification

No premade-label round 1 anymore -- real two-round projection (Kemper-style):

- **Stage A (round 1, broad gate)**: project every AoU sample with ACAF data onto the *whole-cohort* 1000G PCA, fit a loose Mahalanobis ellipsoid per superpopulation (`EUR`, `AFR`), write broad candidate lists.
- **Stage B (round 2, tight fit)**: reproject each superpop's candidates onto *that superpop's own* 1000G PCA (`01_build_1000g_reference.ipynb`'s per-superpop fits -- PC1/PC2 there actually carry within-population structure, unlike the whole-cohort fit), fit the tight per-`SAMPLE_SET` Mahalanobis ellipsoid there.

Reads `02_build_ancestry_panel_hm3.ipynb`'s single whole-cohort HM3 panel. Writes `final_keep_ids_{SAMPLE_SET}_{prob_tag}.txt` to `ancestry_panel/final_pca/`.

## Compute resource

8-16 vCPU is plenty. Stage A now processes every AoU sample with ACAF data, not just a premade-label subset -- a real increase in scope over the old design.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os
import subprocess

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

def run_bash(script):
    subprocess.run(["bash", "-c", "set -e\n" + script], check=True)

## Configuration

`SUPERPOP_POPS`: which 1000G populations pool into each broad superpop gate (Stage A) and which get used for the tight ellipsoid (Stage B's `ellipsoid_ref_pops`, a subset). `ROUND1_GATE_THRESHOLD`: loose, deliberately looser than any final `SAMPLE_SET` threshold -- Stage A's job is just "plausibly this continent," not final classification. `eur_unfiltered` = keep everyone Stage A's EUR gate passed, skip Stage B's ellipsoid.

In [ ]:
SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
}
ROUND1_GATE_THRESHOLD = 0.999999   # loose -- Stage A only needs to gate to the right continent

SAMPLE_SETS = {
    "eur":            {"superpop": "EUR", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": 0.999999},
    "eur_stringent":  {"superpop": "EUR", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": 0.99},
    "eur_loose":      {"superpop": "EUR", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": 0.99999999},
    "eur_unfiltered": {"superpop": "EUR", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": None},
    "afr":            {"superpop": "AFR", "ellipsoid_ref_pops": SUPERPOP_POPS["AFR"], "ellipsoid_threshold": 0.999},
}

N_PCS_ELLIPSOID = 2   # Mahalanobis fit dimensionality, matching every prior round's own convention

CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "covariance_v9"
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
KG_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # 01_build_1000g_reference.ipynb's output, CDR-independent
PANEL_PATH = f"{KG_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
KG_OUT_PREFIX = f"{KG_DIR}/1kg_all_qc"

# whole-cohort fit (Stage A) + one fit per superpopulation (Stage B)
KG_PCA_PREFIXES = {"ALL": f"{KG_DIR}/1kg_all_pca"}
for sp in SUPERPOP_POPS:
    KG_PCA_PREFIXES[sp] = f"{KG_DIR}/1kg_{sp.lower()}_pca"
for prefix in KG_PCA_PREFIXES.values():
    for ext in ("eigenvec.allele", "acount"):
        assert os.path.isfile(f"{prefix}.{ext}"), f"missing {prefix}.{ext} -- run 01_build_1000g_reference.ipynb first"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_round2")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

## Inputs

`02_build_ancestry_panel_hm3.ipynb`'s single whole-cohort HM3 panel -- no `BASE_GROUP`, already harmonized against 1000G at build time.

In [ ]:
PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_panel"
MERGED_NAME = f"ancestry_panel_hm3_{CDR_VERSION}"
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

import shutil
for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{PANEL_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing merged ancestry panel: {bucket_path!r} -- run 02_build_ancestry_panel_hm3.ipynb's merge section first"
    )
    if not os.path.isfile(local_path) or os.path.getsize(local_path) != os.path.getsize(bucket_path):
        shutil.copy(bucket_path, local_path)

FINAL_PCA_BUCKET_DIR = f"{PANEL_DIR}/final_pca"
os.makedirs(FINAL_PCA_BUCKET_DIR, exist_ok=True)

print(MERGED_PREFIX)
print(FINAL_PCA_BUCKET_DIR)

## Shared harmonize/project/reproject helpers

Same logic as the single-round notebook this replaces, wrapped as functions so Stage A and Stage B can call it against different 1000G PC spaces (whole-cohort vs. per-superpop) without duplicating the shell pipeline. `harmonize_and_project()`: biallelic-filter the input panel, build an ID+REF+ALT "agreeing" list against `kg_pca_prefix.acount` (bare ID match would silently let `--score` skip allele mismatches), then `--score` onto `kg_pca_prefix`'s weights. `reproject_reference()`: sanity-check by projecting the 1000G reference onto its own PCs, restricted to the same variant set the AoU projection actually used.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2

def harmonize_and_project(panel_prefix, keep_path, kg_pca_prefix, out_prefix):
    biallelic_prefix = f"{out_prefix}_biallelic"
    keep_clause = f'--keep "{keep_path}"' if keep_path else ""
    run_bash(f'''
        plink2 --pfile "{panel_prefix}" {keep_clause} --max-alleles 2 --rm-dup exclude-all --make-pgen --out "{biallelic_prefix}"

        grep -v '^##' "{biallelic_prefix}.pvar" | awk 'NR>1 {{print $3, $4, $5}}' | sort > "{out_prefix}_acaf_id_ref_alt.sorted"
        awk 'NR>1 {{print $2, $3, $4}}' "{kg_pca_prefix}.acount" | sort > "{out_prefix}_kg_id_ref_alt.sorted"
        comm -12 "{out_prefix}_acaf_id_ref_alt.sorted" "{out_prefix}_kg_id_ref_alt.sorted" | awk '{{print $1}}' > "{out_prefix}_agreeing_snps.ids"
        echo "Agreeing with 1000G (ID+REF+ALT): $(wc -l < "{out_prefix}_agreeing_snps.ids")"

        WEIGHTS="{kg_pca_prefix}.eigenvec.allele"
        HEADER=$(head -1 "$WEIGHTS")
        ID_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'ID' | cut -d: -f1)
        A1_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'A1' | cut -d: -f1)
        PC1_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'PC1' | cut -d: -f1)
        PC_LAST_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'PC20' | cut -d: -f1)

        plink2 --pfile "{biallelic_prefix}" --extract "{out_prefix}_agreeing_snps.ids" --nonfounders \\
          --read-freq "{kg_pca_prefix}.acount" \\
          --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize list-variants \\
          --score-col-nums "${{PC1_COL}}-${{PC_LAST_COL}}" \\
          --out "{out_prefix}"
    ''')

def reproject_reference(kg_out_prefix, kg_pca_prefix, projected_out):
    reproject_out = f"{projected_out}_kg_reprojected"
    run_bash(f'''
        WEIGHTS="{kg_pca_prefix}.eigenvec.allele"
        HEADER=$(head -1 "$WEIGHTS")
        ID_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'ID' | cut -d: -f1)
        A1_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'A1' | cut -d: -f1)
        PC1_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'PC1' | cut -d: -f1)
        PC_LAST_COL=$(echo "$HEADER" | tr '\\t' '\\n' | grep -nx 'PC20' | cut -d: -f1)

        plink2 --bfile "{kg_out_prefix}" --extract "{projected_out}.sscore.vars" --nonfounders \\
          --read-freq "{kg_pca_prefix}.acount" \\
          --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \\
          --score-col-nums "${{PC1_COL}}-${{PC_LAST_COL}}" \\
          --out "{reproject_out}"
    ''')
    return reproject_out

def load_projections(projected_out, reproject_out, panel):
    aou = pd.read_csv(f"{projected_out}.sscore", sep=r"\s+")
    aou_id_col = "#IID" if "#IID" in aou.columns else "IID"
    aou = aou.rename(columns={aou_id_col: "sample"})

    ref = pd.read_csv(f"{reproject_out}.sscore", sep=r"\s+")
    ref_id_col = "#IID" if "#IID" in ref.columns else "IID"
    ref = ref.rename(columns={ref_id_col: "sample"}).merge(panel[["sample", "pop", "super_pop"]], on="sample")
    return aou, ref

def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)

def fit_ellipsoid(aou_df, ref_df, ref_pop_list, threshold_quantile, n_pcs=N_PCS_ELLIPSOID):
    pc_cols = [f"PC{i}_AVG" for i in range(1, n_pcs + 1)]
    ref_mask = ref_df["pop"].isin(ref_pop_list)
    ref_pcs = ref_df.loc[ref_mask, pc_cols].values
    assert len(ref_pcs) > n_pcs, f"too few reference samples ({len(ref_pcs)}) for {ref_pop_list}"

    mean = ref_pcs.mean(axis=0)
    cov_inv = np.linalg.inv(np.cov(ref_pcs, rowvar=False))
    threshold = np.sqrt(chi2.ppf(threshold_quantile, df=n_pcs))

    aou_mahal = np.array([mahal(row, mean, cov_inv) for row in aou_df[pc_cols].values])
    keep_mask = aou_mahal <= threshold

    ref_retention = ref_df.loc[ref_mask, pc_cols].apply(
        lambda row: mahal(row.values, mean, cov_inv) <= threshold, axis=1
    ).mean()
    return keep_mask, ref_retention

panel = pd.read_csv(PANEL_PATH, sep="\t")

## Stage A: broad gate (round 1)

Project every AoU sample onto the whole-cohort 1000G PCA, fit a loose ellipsoid per superpopulation, write `round1_candidates_{superpop}.txt`.

In [ ]:
STAGE_A_PROJECTED = os.path.join(LOCAL_WORK_DIR, "stage_a_projected")

harmonize_and_project(MERGED_PREFIX, None, KG_PCA_PREFIXES["ALL"], STAGE_A_PROJECTED)
stage_a_reproject_out = reproject_reference(KG_OUT_PREFIX, KG_PCA_PREFIXES["ALL"], STAGE_A_PROJECTED)
stage_a_aou, stage_a_ref = load_projections(STAGE_A_PROJECTED, stage_a_reproject_out, panel)

round1_candidates = {}
for superpop, ref_pops in SUPERPOP_POPS.items():
    keep_mask, ref_retention = fit_ellipsoid(stage_a_aou, stage_a_ref, ref_pops, ROUND1_GATE_THRESHOLD)
    candidates = stage_a_aou.loc[keep_mask, "sample"]
    round1_candidates[superpop] = candidates
    print(f"[Stage A: {superpop}] reference retention {ref_retention:.1%}, {len(candidates):,}/{len(stage_a_aou):,} AoU candidates")

    candidates_path = os.path.join(LOCAL_WORK_DIR, f"round1_candidates_{superpop.lower()}.txt")
    candidates.to_csv(candidates_path, index=False, header=False)

### Stage A plot

Whole-cohort projection, colored by which superpop gate each AoU sample passed ("none" if it passed no gate).

In [ ]:
import matplotlib.pyplot as plt

gate_label = pd.Series("none", index=stage_a_aou.index)
for superpop in SUPERPOP_POPS:
    passed = stage_a_aou["sample"].isin(round1_candidates[superpop])
    gate_label = gate_label.where(~passed, superpop)

fig, ax = plt.subplots(figsize=(7, 6))
for label, grp in stage_a_aou.groupby(gate_label):
    ax.scatter(grp["PC1_AVG"], grp["PC2_AVG"], s=3, alpha=0.3, label=f"{label} (n={len(grp):,})")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Stage A: whole-cohort projection, colored by superpop gate")
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

## Stage B: tight fit within the matched superpopulation (round 2)

For each `SAMPLE_SET`, reprojects its superpop's Stage A candidates onto that superpop's own 1000G PCA, fits the tight ellipsoid at `ellipsoid_threshold`. `eur_unfiltered` skips the ellipsoid, keeps all of Stage A's EUR candidates.

In [ ]:
stage_b_cache = {}   # superpop -> (aou_df, ref_df), reused across SAMPLE_SETs sharing a superpop

for sample_set, cfg in SAMPLE_SETS.items():
    superpop = cfg["superpop"]

    if superpop not in stage_b_cache:
        candidates_path = os.path.join(LOCAL_WORK_DIR, f"round1_candidates_{superpop.lower()}.txt")
        stage_b_projected = os.path.join(LOCAL_WORK_DIR, f"stage_b_projected_{superpop.lower()}")
        harmonize_and_project(MERGED_PREFIX, candidates_path, KG_PCA_PREFIXES[superpop], stage_b_projected)
        stage_b_reproject_out = reproject_reference(KG_OUT_PREFIX, KG_PCA_PREFIXES[superpop], stage_b_projected)
        stage_b_cache[superpop] = load_projections(stage_b_projected, stage_b_reproject_out, panel)

    aou_df, ref_df = stage_b_cache[superpop]
    threshold_quantile = cfg["ellipsoid_threshold"]

    if threshold_quantile is None:
        keep_ids = aou_df["sample"]
        prob_tag = "unfiltered"
    else:
        keep_mask, ref_retention = fit_ellipsoid(aou_df, ref_df, cfg["ellipsoid_ref_pops"], threshold_quantile)
        keep_ids = aou_df.loc[keep_mask, "sample"]
        prob_tag = f"p{threshold_quantile * 100:g}"
        print(f"[{sample_set}] {'+'.join(cfg['ellipsoid_ref_pops'])} reference retention: {ref_retention:.1%}")

    keep_path = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_keep_ids_{sample_set}_{prob_tag}.txt")
    keep_ids.to_csv(keep_path, index=False, header=False)
    print(f"[{sample_set}] {len(keep_ids):,}/{len(aou_df):,} retained -> {keep_path}")

### Stage B plot (EUR family)

EUR candidates reprojected onto the EUR-specific 1000G PCA -- confirm within-EUR structure is visible on PC1/PC2 now, unlike the whole-cohort fit.

In [ ]:
eur_aou, eur_ref = stage_b_cache["EUR"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(eur_aou["PC1_AVG"], eur_aou["PC2_AVG"], s=4, alpha=0.2, color="tab:blue")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_title(f"Stage A EUR candidates, projected onto EUR-specific 1000G PCA (n={len(eur_aou):,})")

for pop, grp in eur_ref.groupby("pop"):
    axes[1].scatter(grp["PC1_AVG"], grp["PC2_AVG"], s=8, alpha=0.7, label=pop)
axes[1].scatter(eur_aou["PC1_AVG"], eur_aou["PC2_AVG"], s=3, alpha=0.1, color="black", label="AoU EUR candidates")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("vs. reprojected EUR-specific 1000G reference")
axes[1].legend(markerscale=3, fontsize=8, loc="best")

plt.tight_layout()
plt.show()

## Next steps

`04_final_pca.ipynb` reads these keep-lists and refits to 10 PCs within each `SAMPLE_SET`. `05_genome_wide_qc_thinning_batch_submit.ipynb` reads them too, to build the GRM panel per `SAMPLE_SET`.